# Physics-Law Credibility Benchmark

Controlled audit for Beer-Lambert and Moens-Korteweg law completeness using frozen activation probes.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "analysis" / "physics_law_credibility.py").exists():
    ROOT = ROOT / "supervised_baselines"
sys.path.insert(0, str(ROOT / "analysis"))

from physics_law_credibility import (
    LawAuditConfig,
    build_beer_dataset,
    build_mk_dataset,
    export_summary,
    generate_beer_lambert_sample,
    generate_moens_korteweg_sample,
    run_beer_audit,
    run_mk_audit,
    set_seed,
)

QUICK = True  # set False for publication run (3 seeds, full data)
SEEDS = [0] if QUICK else [0, 1, 2]
OUTPUT = ROOT / "results" / "physics_credibility"

## Beer–Lambert

Modified Beer–Lambert: \(I_\lambda(t) = I_0 \exp(-\mu_\lambda c L(t))\).

Sufficient statistic for SpO₂: ratio-of-ratios \(R = (AC_{red}/DC_{red}) / (AC_{ir}/DC_{ir})\).

**Unanswerable** when a wavelength is missing or wavelengths come from mismatched subjects.

In [ ]:
rng = np.random.default_rng(0)
sample = generate_beer_lambert_sample(rng, "clean")
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(sample.red, label="660 nm")
ax[0].plot(sample.ir, label="940 nm")
ax[0].set(title=f"Beer-Lambert waveforms (SpO2={sample.spo2:.1f}%)", xlabel="sample", ylabel="intensity")
ax[0].legend()
ax[1].bar(["R ratio", "AC red", "AC ir"], [sample.ratio_r, sample.ac_red, sample.ac_ir])
ax[1].set(title="Law statistics")
fig.tight_layout()
plt.show()

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() and not QUICK else "cpu")
cfg = LawAuditConfig(
    seed=0,
    n_train=64 if QUICK else 400,
    n_probe=24 if QUICK else 120,
    n_val=24 if QUICK else 120,
    n_test=16 if QUICK else 80,
    epochs=5 if QUICK else 120,
    n_bootstrap=3 if QUICK else 15,
)
beer_report = run_beer_audit(cfg, device)
cross = beer_report["models"]["cross_attention"]
print("Clean MAE:", cross["scenario_mae"].get("clean"))
print("Probe R2:", cross["probe_r2"])
print("Answerability AUROC:", cross["answerability_all"]["auroc"])
print("Probe AURC:", cross["selective"]["probe"]["aurc"], "vs random:", cross["selective"]["random"]["aurc"])
print("Decision:", beer_report["decision"])

## Moens–Korteweg

\(PWV = \sqrt{Eh/(\rho D)}\), \(PTT = L/PWV\) (ms).

Pressure–stiffness: \(E(P) = E_0 \exp(\alpha P/100)\).

**BP is not uniquely determined by PTT alone** without subject-specific calibration (path length, stiffness parameters).

In [ ]:
mk = generate_moens_korteweg_sample(np.random.default_rng(1), "clean")
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(mk.proximal, label="proximal")
ax[0].plot(mk.distal, label="distal")
ax[0].set(title=f"MK waveforms (PTT={mk.ptt_ms:.1f} ms, BP={mk.bp:.0f} mmHg)", xlabel="sample")
ax[0].legend()
ax[1].bar(["PTT ms", "PWV m/s", "BP"], [mk.ptt_ms, mk.pwv, mk.bp])
ax[1].set(title="Law statistics")
fig.tight_layout()
plt.show()

In [ ]:
mk_report = run_mk_audit(cfg, device)
cross_mk = mk_report["models"]["cross_attention"]
print("Clean MAE:", cross_mk["scenario_mae"].get("clean"))
print("Probe R2:", cross_mk["probe_r2"])
print("Answerability AUROC:", cross_mk["answerability_all"]["auroc"])
print("Intervention:", mk_report.get("intervention"))
print("Decision:", mk_report["decision"])

In [ ]:
reports = []
for seed in SEEDS:
    set_seed(seed)
    c = LawAuditConfig(seed=seed, **{k: getattr(cfg, k) for k in cfg.__dataclass_fields__ if k != "seed"})
    reports.append(run_beer_audit(c, device))
    reports.append(run_mk_audit(c, device))

summary_path = export_summary(reports, OUTPUT, {"quick": QUICK, "seeds": SEEDS})
print("Wrote", summary_path)
print("Open report: analysis/credibility_report/index.html (serve with python -m http.server 8000)")

## Interpretation

- **Decodable law statistic** (e.g. R or PTT in activations) ≠ model uses it causally.
- **Probe abstention beats random** → external credibility monitor is useful.
- **Negative result** (decodable but no selective improvement) should be reported honestly.

This benchmark does **not** validate clinical pulse oximetry or cuffless BP. See `analysis/credibility_report/index.html` for the interactive audit walkthrough.

# When a model should abstain: two physiological laws

This tutorial tests a narrow credibility claim: **a frozen readout of internal activations can detect when information required by a stated physical law is absent or violated, and abstention can then lower error.**

We use two transparent synthetic systems:

1. modified Beer–Lambert pulse oximetry (red + infrared → SpO₂), and
2. Moens–Korteweg pulse-wave velocity with explicit subject calibration (proximal + distal pulse → BP).

This is a controlled identifiability benchmark—not clinical validation, intrinsic awareness, or a claim that these simulators are individually novel. Prior work already studies synthetic oximetry, simulated PTT/BP, physics-informed learning, and uncertainty.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "analysis":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "analysis"))

from physics_law_credibility import (
    BEER_EQUATION, MK_EQUATIONS,
    simulate_beer_lambert, simulate_moens_korteweg,
    run_benchmark,
)

QUICK = True  # Set False for the multi-seed publication configuration.
OUTPUT_DIR = ROOT / "results" / "physics_credibility" / "tutorial_quick"
print(f"Mode: {'quick tutorial' if QUICK else 'publication'}")

## 1. Modified Beer–Lambert pulse oximetry

For each wavelength, intensity is generated from

\[
I_\lambda(t)=G_\lambda\exp[-A_{tissue,\lambda}-(\epsilon_{oxy,\lambda}c_{oxy}+\epsilon_{deoxy,\lambda}c_{deoxy})L_\lambda(t)].
\]

The model receives red (660 nm), infrared (940 nm), and calibration context. The interpretable sufficient statistic is the ratio of ratios

\[
R=\frac{AC_{red}/DC_{red}}{AC_{IR}/DC_{IR}}.
\]

Missing a wavelength or pairing wavelengths from different virtual patients makes the stated inverse problem non-identifiable. Pigmentation and motion shifts remain separately labeled rather than being called mathematically unknowable.

In [ ]:
beer = simulate_beer_lambert(3, "clean", seed=7)
t = np.arange(beer.stream_a.shape[1]) / 25.0
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t, beer.stream_a[0, :, 0], label="Red, 660 nm")
ax.plot(t, beer.stream_b[0, :, 0], label="Infrared, 940 nm")
ax.set(xlabel="Time (s)", ylabel="Normalized intensity (a.u.)",
       title=f"Identifiable example: SpO₂={beer.target[0]:.1f}%, R={beer.statistics['ratio_r'][0]:.2f}")
ax.legend(); plt.show()
print(BEER_EQUATION)

In [ ]:
report = run_benchmark(
    output_dir=OUTPUT_DIR,
    quick=QUICK,
    seeds=(0, 1, 2),
    device="cuda" if __import__("torch").cuda.is_available() else "cpu",
)
beer_result = report["laws"]["beer_lambert"]["aggregate"]
print("Beer–Lambert audit")
print(f"  R probe R²:             {beer_result['core_statistic_r2']:.3f}")
print(f"  Answerability AUROC:    {beer_result['answerability_auroc']:.3f}")
print(f"  Held-out violation AUC: {beer_result['heldout_violation_auroc']:.3f}")
print(f"  Probe / random AURC:    {beer_result['probe_aurc']:.3f} / {beer_result['random_aurc']:.3f}")
print(f"  All criteria pass:      {beer_result['all_criteria_pass']}")

## 2. Moens–Korteweg, PTT, and calibrated BP

The simulator uses

\[
PWV=\sqrt{\frac{Eh}{\rho D}},\qquad PTT=\frac{L}{PWV},
\]

plus an explicit virtual-subject calibration

\[
E(P)=E_0\exp[\alpha(P-100\,\mathrm{mmHg})].
\]

The model receives proximal and distal pulse waves plus path length, anatomy, and stiffness calibration. If path length or stiffness calibration is absent, the same PTT can correspond to multiple pressures. Therefore **PTT alone does not uniquely determine BP**.

In [ ]:
mk = simulate_moens_korteweg(3, "clean", seed=11)
t = np.arange(mk.stream_a.shape[1]) / 100.0
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t, mk.stream_a[0, :, 0], label="Proximal pulse")
ax.plot(t, mk.stream_b[0, :, 0], label="Distal pulse")
ax.set(xlabel="Time (s)", ylabel="Pulse amplitude (a.u.)",
       title=f"BP={mk.target[0]:.1f} mmHg, PTT={mk.statistics['ptt_ms'][0]:.1f} ms, PWV={mk.statistics['pwv_m_s'][0]:.1f} m/s")
ax.legend(); plt.show()
print(MK_EQUATIONS)

In [ ]:
mk_result = report["laws"]["moens_korteweg"]["aggregate"]
print("Moens–Korteweg audit")
print(f"  PTT probe R²:           {mk_result['core_statistic_r2']:.3f}")
print(f"  Answerability AUROC:    {mk_result['answerability_auroc']:.3f}")
print(f"  Held-out violation AUC: {mk_result['heldout_violation_auroc']:.3f}")
print(f"  Probe / random AURC:    {mk_result['probe_aurc']:.3f} / {mk_result['random_aurc']:.3f}")
print(f"  All criteria pass:      {mk_result['all_criteria_pass']}")

In [ ]:
for name, result in [("Beer–Lambert", beer_result), ("Moens–Korteweg", mk_result)]:
    print(name)
    print(f"  Probe-aligned intervention ΔMAE: {result['probe_intervention_delta_mae']:.4f}")
    print(f"  Random-direction intervention:   {result['random_intervention_delta_mae']:.4f}")
    print(f"  Credibility decision:            {result['decision']}")
    print()

print("Interpretation: decodability without selective-risk or intervention evidence is a mixed/negative result, not proof that the model uses the law.")

## What this establishes—and what it does not

A successful run requires five independent checks: identifiable-data accuracy, law-statistic decodability, matched and held-out violation detection, lower selective risk, and greater sensitivity to probe-aligned than random interventions.

The contribution is the **cross-law audit protocol**, not the individual simulators. The optical model omits full photon transport; the arterial model omits distributed and viscoelastic hemodynamics except where used as an explicit stress test. Synthetic success must be followed by real-data validation and must not be interpreted as regulatory or clinical safety evidence.

The machine-readable report is written to `results/physics_credibility/.../summary.json`. The companion HTML page turns that report into a model-credibility walkthrough.